# EEG Seizure Analysis: Signal Processing, Feature Engineering & Exploratory Classification

**Project type:** Reproducible exploratory computational-neuroscience project  
**Dataset:** CHB-MIT Scalp EEG Database (PhysioNet, version 1.0.0)  
**Primary question:** Can time- and frequency-domain features of scalp EEG distinguish annotated seizure windows from non-seizure windows in selected recordings?

## Scope and scientific guardrails
This notebook is an educational, exploratory analysis—not a clinical device, diagnostic tool, or validated seizure detector. It uses a small, deliberately limited subset of one subject's recordings. Results cannot establish generalization to other people, hospitals, recording systems, or clinical settings. We will avoid claiming clinical performance.

**Selected recordings**
- `chb01_03.edf`: contains a seizure annotated from 2996 to 3036 seconds.
- `chb01_01.edf`: a non-seizure recording according to the subject summary.

The PhysioNet subject summary is the source of seizure timing. The dataset contains 23 EEG channels and uses a 256 Hz sampling rate for this subject.

**Important evaluation limitation:** Because the selected recordings are from the same subject and only two files, this notebook uses a simple exploratory train/test split. This is not a subject-independent or clinically meaningful validation. A stronger follow-up should include multiple subjects and hold out entire subjects or recordings.

## 1. Environment and reproducibility

Run this notebook in Google Colab with a CPU runtime. The notebook downloads two EDF files (roughly 80 MB total) and the CHB01 summary file directly from PhysioNet.

In [ ]:
!pip -q install mne scipy scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import os, re, io, warnings, requests, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne

from scipy.signal import welch
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, average_precision_score, precision_recall_curve
)

SEED = 42
np.random.seed(SEED)
mne.set_log_level("ERROR")

PROJECT_DIR = Path("/content/eeg_seizure_project")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "tableau_exports"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("MNE:", mne.__version__)
print("Project directory:", PROJECT_DIR)

## 2. Download the selected public recordings

The full CHB-MIT dataset is large (about 42.6 GB uncompressed). We only download two recordings and one summary file for this focused exercise. PhysioNet provides the dataset under the Open Data Commons Attribution License v1.0; cite the dataset in any public presentation.

In [ ]:
BASE = "https://physionet.org/files/chbmit/1.0.0/chb01/"
FILES = ["chb01_01.edf", "chb01_03.edf", "chb01-summary.txt"]

def download_if_missing(filename):
    dest = DATA_DIR / filename
    if dest.exists() and dest.stat().st_size > 0:
        print(f"Already present: {filename} ({dest.stat().st_size/1e6:.1f} MB)")
        return dest
    url = BASE + filename
    print("Downloading", url)
    with requests.get(url, stream=True, timeout=180) as response:
        response.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
    print(f"Saved {filename}: {dest.stat().st_size/1e6:.1f} MB")
    return dest

for filename in FILES:
    download_if_missing(filename)

## 3. Read the subject summary and verify annotations

The summary text records seizure onset and offset relative to the beginning of each EDF file. We parse the file rather than hard-code annotation times into the feature extraction logic.

In [ ]:
summary_path = DATA_DIR / "chb01-summary.txt"
summary_text = summary_path.read_text(errors="replace")

# Extract per-file seizure intervals from the official summary.
blocks = re.split(r"(?=File Name:)", summary_text)
annotation_rows = []
for block in blocks:
    file_match = re.search(r"File Name:\s*(\S+)", block)
    if not file_match:
        continue
    filename = file_match.group(1)
    starts = [int(x) for x in re.findall(r"Seizure Start Time:\s*(\d+)\s*seconds", block)]
    ends = [int(x) for x in re.findall(r"Seizure End Time:\s*(\d+)\s*seconds", block)]
    for idx, (start, end) in enumerate(zip(starts, ends), start=1):
        annotation_rows.append({
            "recording": filename, "seizure_number": idx,
            "seizure_start_sec": start, "seizure_end_sec": end
        })

annotations = pd.DataFrame(annotation_rows)
annotations

In [ ]:
# Verify expected annotation for chb01_03 and show selected recordings.
selected_annotations = annotations[annotations["recording"].isin(["chb01_01.edf", "chb01_03.edf"])]
display(selected_annotations)

assert (selected_annotations["recording"] == "chb01_03.edf").any(), "Expected seizure annotation not found."
assert not (selected_annotations["recording"] == "chb01_01.edf").any(), "Unexpected seizure annotation in chb01_01."
print("Summary parsing complete.")

## 4. Load EDF files and inspect metadata

EEG is measured in volts by MNE. For plots and features we convert to microvolts (µV), a more readable scale. We exclude non-EEG/dummy channels if present and retain channels with EEG-like names.

In [ ]:
recordings = {}
for filename in ["chb01_01.edf", "chb01_03.edf"]:
    raw = mne.io.read_raw_edf(DATA_DIR / filename, preload=True, verbose="ERROR")
    # Exclude channels whose names are '-' or empty; keep EEG channels.
    keep = [ch for ch in raw.ch_names if ch.strip() not in ["", "-"]]
    raw.pick(keep)
    recordings[filename] = raw
    print(
        filename,
        "| sampling rate:", raw.info["sfreq"],
        "| channels:", len(raw.ch_names),
        "| duration (min):", round(raw.times[-1] / 60, 2)
    )
    print("First channels:", raw.ch_names[:8])

## 5. Visualize raw EEG around the annotated seizure

The seizure in `chb01_03.edf` is annotated from 2996 to 3036 seconds. We plot a short interval before, during, and after this event. The displayed channels are vertically offset only to make the traces easier to distinguish.

In [ ]:
seizure_start, seizure_end = 2996, 3036
raw_seizure = recordings["chb01_03.edf"]
sfreq = raw_seizure.info["sfreq"]

plot_start = seizure_start - 20
plot_end = seizure_end + 20
segment = raw_seizure.copy().crop(tmin=plot_start, tmax=plot_end)
data_uv = segment.get_data() * 1e6
times = segment.times + plot_start

n_plot = min(6, data_uv.shape[0])
offset = 250
plt.figure(figsize=(15, 8))
for i in range(n_plot):
    plt.plot(times, data_uv[i] + i*offset, lw=0.7, label=segment.ch_names[i])
plt.axvspan(seizure_start, seizure_end, alpha=0.18, label="Annotated seizure")
plt.xlabel("Time from recording start (seconds)")
plt.ylabel("Amplitude (µV, vertically offset)")
plt.title("CHB01_03: EEG around annotated seizure interval")
plt.legend(loc="upper right", ncol=2)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 6. Preprocessing and window-level feature extraction

We use non-overlapping 10-second windows. For each window, we calculate features from the average across retained EEG channels:
- mean and standard deviation of amplitude;
- root mean square (RMS);
- line length (a simple measure of signal variation);
- relative band power for delta, theta, alpha, beta, and gamma bands;
- spectral entropy.

The seizure label is 1 when a window overlaps an annotated seizure interval; otherwise it is 0. Windows that partially overlap a seizure are labelled seizure. For a production study, boundary handling and expert review would need more careful treatment.

We apply a 1–40 Hz bandpass filter for feature extraction. This means our gamma feature represents only the 30–40 Hz portion of the conventional gamma band.

In [ ]:
BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma_30_40": (30, 40)
}
WINDOW_SEC = 10

def spectral_features(x, fs):
    freqs, psd = welch(x, fs=fs, nperseg=min(len(x), int(fs*4)))
    total = np.trapz(psd, freqs)
    feats = {}
    for name, (lo, hi) in BANDS.items():
        mask = (freqs >= lo) & (freqs < hi)
        power = np.trapz(psd[mask], freqs[mask]) if mask.sum() > 1 else 0.0
        feats[f"rel_power_{name}"] = power / total if total > 0 else np.nan
    p = psd / np.sum(psd) if np.sum(psd) > 0 else np.zeros_like(psd)
    p = p[p > 0]
    feats["spectral_entropy"] = -np.sum(p * np.log2(p)) if len(p) else np.nan
    return feats

def extract_recording_features(filename, raw, seizure_intervals):
    fs = raw.info["sfreq"]
    filtered = raw.copy().filter(l_freq=1.0, h_freq=40.0, verbose="ERROR")
    n_samples = int(WINDOW_SEC * fs)
    n_total = filtered.n_times
    rows = []

    for start_sample in range(0, n_total - n_samples + 1, n_samples):
        stop_sample = start_sample + n_samples
        start_sec = start_sample / fs
        end_sec = stop_sample / fs
        # Average across channels after filtering; features are computed on this
        # aggregate signal to keep the exported Tableau table compact.
        x_channels = filtered.get_data(start=start_sample, stop=stop_sample) * 1e6
        x = np.mean(x_channels, axis=0)

        label = int(any(
            start_sec < event_end and end_sec > event_start
            for event_start, event_end in seizure_intervals
        ))

        row = {
            "recording": filename,
            "subject": "chb01",
            "window_start_sec": start_sec,
            "window_end_sec": end_sec,
            "window_mid_sec": (start_sec + end_sec) / 2,
            "seizure_label": label,
            "seizure_status": "Seizure" if label else "Non-seizure",
            "mean_uv": float(np.mean(x)),
            "std_uv": float(np.std(x)),
            "rms_uv": float(np.sqrt(np.mean(x**2))),
            "line_length": float(np.mean(np.abs(np.diff(x))))
        }
        row.update(spectral_features(x, fs))
        rows.append(row)
    return pd.DataFrame(rows)

interval_map = {
    "chb01_01.edf": [],
    "chb01_03.edf": [(2996, 3036)]
}

feature_frames = []
for filename, raw in recordings.items():
    feature_frames.append(
        extract_recording_features(filename, raw, interval_map[filename])
    )

features = pd.concat(feature_frames, ignore_index=True)
print("Window rows:", len(features))
print("Class counts:")
display(features["seizure_status"].value_counts().rename_axis("class").to_frame("windows"))
display(features.head())

## 7. Explore feature distributions

These plots are descriptive. They help us see whether the selected features differ in this small sample. They do not establish that a feature is a reliable seizure biomarker.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.boxplot(data=features, x="seizure_status", y="rms_uv", ax=axes[0,0])
axes[0,0].set_title("RMS amplitude by window label")
sns.boxplot(data=features, x="seizure_status", y="line_length", ax=axes[0,1])
axes[0,1].set_title("Line length by window label")
sns.boxplot(data=features, x="seizure_status", y="rel_power_theta", ax=axes[1,0])
axes[1,0].set_title("Relative theta power by window label")
sns.boxplot(data=features, x="seizure_status", y="spectral_entropy", ax=axes[1,1])
axes[1,1].set_title("Spectral entropy by window label")
plt.tight_layout()
plt.show()

## 8. Exploratory baseline models

We train Logistic Regression and Random Forest models on a stratified random split of the selected windows. This is strictly a demonstration of a modeling workflow.

**Critical limitation:** Windows from the same two recordings may share subject- and recording-specific characteristics. A random window split can overestimate performance because nearby windows are not independent. Do not report these scores as evidence of clinical or cross-subject performance. A stronger follow-up should use subject-wise or recording-wise holdout and more subjects.

In [ ]:
feature_cols = [
    "mean_uv", "std_uv", "rms_uv", "line_length",
    "rel_power_delta", "rel_power_theta", "rel_power_alpha",
    "rel_power_beta", "rel_power_gamma_30_40", "spectral_entropy"
]
X = features[feature_cols].replace([np.inf, -np.inf], np.nan)
y = features["seizure_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

models = {
    "Logistic Regression": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED)
    ),
    "Random Forest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=300, class_weight="balanced",
            min_samples_leaf=2, random_state=SEED, n_jobs=-1
        )
    )
}

model_results = []
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    model_results.append({
        "model": name,
        "precision": __import__("sklearn.metrics").metrics.precision_score(y_test, pred, zero_division=0),
        "recall": __import__("sklearn.metrics").metrics.recall_score(y_test, pred, zero_division=0),
        "f1": __import__("sklearn.metrics").metrics.f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, prob) if y_test.nunique() == 2 else np.nan,
        "average_precision": average_precision_score(y_test, prob)
    })
    print("\n", name)
    print(classification_report(y_test, pred, target_names=["Non-seizure", "Seizure"], zero_division=0))

model_metrics = pd.DataFrame(model_results)
display(model_metrics)

## 9. Confusion matrix and feature importance

In [ ]:
best_name = "Random Forest"
best_model = fitted_models[best_name]
best_pred = best_model.predict(X_test)

ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred,
    display_labels=["Non-seizure", "Seizure"],
    cmap="Blues", values_format="d"
)
plt.title(f"{best_name}: exploratory test split")
plt.tight_layout()
plt.show()

rf = best_model.named_steps["randomforestclassifier"]
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)
display(importance)

plt.figure(figsize=(9,5))
sns.barplot(data=importance, x="importance", y="feature")
plt.title("Random Forest feature importance (exploratory)")
plt.tight_layout()
plt.show()

## 10. Export Tableau-ready data

The window-level table is the primary Tableau data source. It contains one row per 10-second EEG window and includes labels and features. The event table describes the annotated seizure interval. The model metrics table is a compact comparison table.

**Tableau dashboard suggestion**
1. KPI tiles: number of windows, seizure windows, non-seizure windows, selected recordings.
2. Timeline: window midpoint on columns, seizure label/status on color, recording as filter.
3. Feature comparison: box plots of RMS, line length, theta power, spectral entropy by seizure status.
4. Model comparison: precision, recall, F1, ROC-AUC (clearly label as exploratory random-window split).
5. Annotation context: event start and end time for CHB01_03.

Do not publish raw patient-level clinical details beyond what is necessary. This public dataset is de-identified, but all outputs should still be handled thoughtfully.

In [ ]:
# Export all key tables
window_csv = OUTPUT_DIR / "eeg_window_features_tableau.csv"
event_csv = OUTPUT_DIR / "seizure_events_tableau.csv"
metrics_csv = OUTPUT_DIR / "model_metrics_exploratory.csv"
importance_csv = OUTPUT_DIR / "feature_importance_exploratory.csv"

features.to_csv(window_csv, index=False)
annotations.to_csv(event_csv, index=False)
model_metrics.to_csv(metrics_csv, index=False)
importance.to_csv(importance_csv, index=False)

print("Exported Tableau files:")
for p in [window_csv, event_csv, metrics_csv, importance_csv]:
    print(p.name, f"({p.stat().st_size/1024:.1f} KB)")

## 11. Research interpretation template

Complete this section only after reviewing your actual outputs.

### Observations
- What differences, if any, did you observe in EEG signal amplitude around the annotated event?
- Which extracted features showed the clearest descriptive differences?
- Did the baseline models identify seizure windows? What were the false positives and false negatives?
- Did performance metrics appear unstable because of the small number of seizure windows?

### Limitations
- One subject and two selected recordings only.
- One annotated seizure event in the selected seizure recording.
- Random window split is not independent validation and may inflate results.
- A single averaged signal across channels discards spatial/channel-specific information.
- Simple band-power features do not capture all relevant neural dynamics.
- This is not a clinically validated seizure detector.

### Next steps
- Include more recordings and subjects.
- Use recording-wise and subject-wise evaluation.
- Analyze channels separately and compare spatial patterns.
- Consider event-level detection and false alarms per hour.
- Compare additional models only after establishing a robust validation design.

## 12. Dataset citation

Shoeb, A. H. (2009). *Application of Machine Learning to Epileptic Seizure Onset Detection* [PhD thesis, Massachusetts Institute of Technology]. The CHB-MIT Scalp EEG Database is available via PhysioNet: https://doi.org/10.13026/C2K01R

Database acknowledgement: Children’s Hospital Boston and the Massachusetts Institute of Technology. Cite the dataset and comply with its Open Data Commons Attribution License v1.0.

**Project status:** Exploratory computational analysis. Not for diagnosis, treatment, or clinical decision-making.